In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP3 Disparate-Impact Mitigation Investigation notebook
(Governance addendum, requested by the user after selecting "Investigate bias mitigation now" in
response to BP3 Gate 4/5's real-run-confirmed adverse_impact_ratio_tags = 0.139 flag).
Single consolidated code cell (platform convention). Idempotent - safe to re-run.

Not a numbered gate (Gates 1-6 plus the executive-rollup "Gate 7" are the Master Plan's fixed
governance cycle - PROJECT_STRUCTURE_LOCKED.md does not permit inventing a new gate number). This
is a standalone, additive investigation step, run only after BP3 Gate 5 (Decision Layer &
Reporting) is real-run confirmed. It does NOT retrain BP3's champion, does NOT change its Gate 3-5
real-run-confirmed outputs, and does NOT change its 0.5 default decision threshold. It extends the
already real-run-confirmed Gate 4/5 finding with additional diagnostics a human governance
reviewer needs, using ONLY real data: the full real 163,091-row Gate 5 decision-records file plus
Gate 5's own already-recorded per-group breakdown. Nothing here is estimated, assumed, or
synthesized - see src/models/bp3_fairness_mitigation.py's own module docstring for the full
real-data-only design rationale.

This notebook also independently CROSS-CHECKS the investigation module's back-computed per-group
confusion-matrix detail (which is derived from Gate 5's already-rounded recorded rates) against a
DIRECT recount from the full real decision-records file's own predicted_label/true_label columns -
never trusting the back-computation alone when the real underlying rows are available to verify
against directly.

Reuses src/models/bp3_fairness_mitigation.py (new, this step) for every real diagnostic
computation - HYPER: no fairness-diagnostic logic is duplicated inline here.
"""

import os, sys, json, math, time, warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )
    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)
    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402

import pandas as pd  # noqa: E402
import yaml  # noqa: E402

print = functools.partial(builtins.print, flush=True)

from models.bp3_fairness_mitigation import (  # noqa: E402
    DISPARATE_TREATMENT_DISCLOSURE,
    build_investigation_summary,
)

CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp3_complaint_escalation_prediction" / "artifacts"
assert ARTIFACTS_DIR.exists(), f"[CHECK FAILED] {ARTIFACTS_DIR} not found - run BP3 Gates 1-5 first."

# ============================================================
# SECTION 4: Load BP3's real config + Gate 4/5 artifacts - the flagged finding is read LIVE from
# the real, already-run-confirmed record, never hardcoded or assumed.
# ============================================================
bp3_config_path = CONFIGS_DIR / "bp3_complaint_escalation_prediction.yaml"
with open(bp3_config_path, "r", encoding="utf-8") as f:
    bp3_config = yaml.safe_load(f)

gate4_block = bp3_config.get("gate4_statistical_validation")
assert gate4_block is not None, "[CHECK FAILED] gate4_statistical_validation missing - run BP3 Gate 4 first."
gate5_block = bp3_config.get("gate5_decision_layer")
assert gate5_block is not None, (
    "[CHECK FAILED] gate5_decision_layer missing from configs/bp3_complaint_escalation_prediction.yaml - "
    "run BP3 Gate 5 (Decision Layer & Reporting) for real before this investigation step."
)
CHAMPION_NAME = gate5_block["champion_model"]
assert CHAMPION_NAME == gate4_block["champion_model"], (
    f"[CHECK FAILED] Champion mismatch: Gate 5 recorded '{CHAMPION_NAME}' but Gate 4's config block "
    f"says '{gate4_block['champion_model']}' - these must agree; re-run Gate 4/5."
)
CONFIG_ADVERSE_IMPACT_RATIO = float(gate4_block["adverse_impact_ratio_tags"])
print(f"[OK] Champion (live, re-verified against Gate 4 + Gate 5): {CHAMPION_NAME}")
print(f"[OK] Config-recorded adverse_impact_ratio_tags: {CONFIG_ADVERSE_IMPACT_RATIO}")

gate5_json_path = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
assert gate5_json_path.exists(), f"[CHECK FAILED] {gate5_json_path} not found - run BP3 Gate 5 first."
with open(gate5_json_path, "r", encoding="utf-8") as f:
    gate5_summary = json.load(f)

disparate_impact_block = gate5_summary.get("disparate_impact_check")
assert (
    disparate_impact_block is not None
), "[CHECK FAILED] disparate_impact_check missing from gate5_decision_layer_summary.json."
assert disparate_impact_block["flagged"] is True, (
    "[CHECK FAILED] gate5_decision_layer_summary.json no longer reports the finding as flagged - "
    "this investigation notebook is written against the flagged state; re-check before re-running."
)
JSON_ADVERSE_IMPACT_RATIO = float(disparate_impact_block["adverse_impact_ratio_recomputed"])
assert abs(JSON_ADVERSE_IMPACT_RATIO - CONFIG_ADVERSE_IMPACT_RATIO) < 1e-9, (
    f"[CHECK FAILED] adverse_impact_ratio mismatch between config yaml ({CONFIG_ADVERSE_IMPACT_RATIO}) "
    f"and gate5_decision_layer_summary.json ({JSON_ADVERSE_IMPACT_RATIO})."
)
TAGS_GROUP_BREAKDOWN = disparate_impact_block["tags_group_breakdown"]
print(
    f"[OK] Real flagged finding confirmed live: adverse_impact_ratio_tags = {JSON_ADVERSE_IMPACT_RATIO} "
    f"across {len(TAGS_GROUP_BREAKDOWN)} real tags_group values."
)

# ============================================================
# SECTION 5: Load the FULL real Gate 5 decision-records file (163,091 rows) - the investigation's
# per-group calibration and threshold-simulation diagnostics need the real row-level data, not
# just Gate 5's already-recorded summary rates.
# ============================================================
decision_records_path = ARTIFACTS_DIR / "gate5_decision_records.csv"
assert (
    decision_records_path.exists()
), f"[CHECK FAILED] {decision_records_path} not found - run BP3 Gate 5 first."

assert_within_ram_ceiling(RESOURCE_LIMITS)
t0 = time.perf_counter()
decision_records_df = pd.read_csv(
    decision_records_path,
    usecols=["true_label", "predicted_label", "predicted_probability", "tags_group"],
    dtype={
        "true_label": "int64",
        "predicted_label": "int64",
        "predicted_probability": "float64",
        "tags_group": "object",
    },
)
load_seconds = time.perf_counter() - t0
print(
    f"[OK] Loaded real gate5_decision_records.csv: {len(decision_records_df):,} rows in {load_seconds:.1f}s"
)

N_DECISION_RECORDS_RECORDED = int(gate5_block["n_decision_records"])
assert len(decision_records_df) == N_DECISION_RECORDS_RECORDED, (
    f"[CHECK FAILED] Loaded {len(decision_records_df):,} rows but Gate 5's config block records "
    f"n_decision_records={N_DECISION_RECORDS_RECORDED:,} - the decision-records file does not match "
    "the real-run-confirmed Gate 5 record; investigate before proceeding."
)
assert set(decision_records_df["true_label"].unique()).issubset(
    {0, 1}
), "[CHECK FAILED] true_label contains values outside {0, 1} - not the expected real binary target."
REAL_TAGS_GROUPS = set(decision_records_df["tags_group"].unique())
BREAKDOWN_TAGS_GROUPS = {row["tags_group"] for row in TAGS_GROUP_BREAKDOWN}
assert REAL_TAGS_GROUPS == BREAKDOWN_TAGS_GROUPS, (
    f"[CHECK FAILED] tags_group values in the real decision-records file ({sorted(REAL_TAGS_GROUPS)}) "
    f"do not match Gate 5's recorded breakdown groups ({sorted(BREAKDOWN_TAGS_GROUPS)})."
)
print("[OK] Row count and tags_group values match Gate 5's real-run-confirmed record exactly.")

# ============================================================
# SECTION 6: Run the real investigation (src/models/bp3_fairness_mitigation.py)
# ============================================================
generated_at = datetime.now(timezone.utc).isoformat()
investigation_summary = build_investigation_summary(
    gate5_summary=gate5_summary,
    decision_records_df=decision_records_df,
    reference_group="NO_TAG",
)
diagnosis = investigation_summary["diagnosis"]
print("\n[INVESTIGATION] Diagnosis verdict:")
print(diagnosis["verdict"])
print(
    f"\n[INVESTIGATION] recall_ratio_min_over_max={diagnosis['recall_ratio_min_over_max']:.4f} "
    f"(balanced={diagnosis['recall_balanced_four_fifths_style']}), "
    f"false_positive_rate_ratio_min_over_max={diagnosis['false_positive_rate_ratio_min_over_max']:.4f} "
    f"(balanced={diagnosis['false_positive_rate_balanced_four_fifths_style']})"
)


# ============================================================
# SECTION 7: INDEPENDENT CROSS-CHECK - the investigation module back-computes each group's
# confusion-matrix detail from Gate 5's already-rounded recorded rates (selection_rate/recall to 4
# decimal places). Since the full real row-level data is now loaded, directly recount each group's
# real TP/FP/TN/FN from predicted_label vs true_label and confirm the back-computed detail is
# consistent with the real underlying rows - never trusting the back-computation alone when the
# real rows are available to check it against.
# ============================================================
def _tolerance(n: int) -> int:
    # Source rates are rounded to 4 decimal places in the real artifact, so the back-computed
    # integer count can differ from the real recount by up to ~0.00005 * n plus rounding slack.
    return max(2, math.ceil(n * 0.0001) + 1)


back_computed_detail = {g["tags_group"]: g for g in investigation_summary["group_confusion_detail"]}
direct_cross_check = {}
direct_cross_check_all_within_tolerance = True
for group_name, sub in decision_records_df.groupby("tags_group", observed=True):
    n = len(sub)
    tp = int(((sub["predicted_label"] == 1) & (sub["true_label"] == 1)).sum())
    fp = int(((sub["predicted_label"] == 1) & (sub["true_label"] == 0)).sum())
    tn = int(((sub["predicted_label"] == 0) & (sub["true_label"] == 0)).sum())
    fn = int(((sub["predicted_label"] == 0) & (sub["true_label"] == 1)).sum())
    bc = back_computed_detail[group_name]
    tol = _tolerance(n)
    diffs = {
        "tp": abs(tp - bc["n_true_positive"]),
        "fp": abs(fp - bc["n_false_positive"]),
        "tn": abs(tn - bc["n_true_negative"]),
        "fn": abs(fn - bc["n_false_negative"]),
    }
    within_tolerance = all(d <= tol for d in diffs.values())
    direct_cross_check_all_within_tolerance &= within_tolerance
    direct_cross_check[group_name] = {
        "n_rows": n,
        "direct_recount": {
            "n_true_positive": tp,
            "n_false_positive": fp,
            "n_true_negative": tn,
            "n_false_negative": fn,
        },
        "back_computed_from_gate5_rates": {
            "n_true_positive": bc["n_true_positive"],
            "n_false_positive": bc["n_false_positive"],
            "n_true_negative": bc["n_true_negative"],
            "n_false_negative": bc["n_false_negative"],
        },
        "abs_diffs": diffs,
        "tolerance": tol,
        "within_tolerance": within_tolerance,
        "direct_false_positive_rate": fp / (fp + tn) if (fp + tn) else float("nan"),
        "direct_recall": tp / (tp + fn) if (tp + fn) else float("nan"),
    }
    status = "[PASS]" if within_tolerance else "[FAIL]"
    direct_fpr = direct_cross_check[group_name]["direct_false_positive_rate"]
    print(
        f"{status} cross-check {group_name}: direct FPR={direct_fpr:.4f} "
        f"vs back-computed FPR={bc['false_positive_rate']:.4f}; "
        f"max abs diff={max(diffs.values())} (tolerance={tol})"
    )

assert direct_cross_check_all_within_tolerance, (
    "[CHECK FAILED] One or more groups' direct recount from the real decision-records file disagrees "
    "with the investigation module's back-computed detail beyond rounding tolerance - do not trust "
    "the diagnosis until this is resolved."
)
print(
    f"\n[OK] Independent cross-check PASSED for all {len(direct_cross_check)} real tags_group values - "
    "the back-computed diagnosis is confirmed consistent with the real underlying decision rows."
)

# ============================================================
# SECTION 8: Write the real output artifact
# ============================================================
output_record = {
    "bp_id": "bp3",
    "investigation": "disparate_impact_mitigation_investigation",
    "requested_by_user_action": "Investigate bias mitigation now (AskUserQuestion selection)",
    "source_adverse_impact_ratio_tags": JSON_ADVERSE_IMPACT_RATIO,
    "source_flagged": disparate_impact_block["flagged"],
    "champion_model": CHAMPION_NAME,
    "n_decision_records": len(decision_records_df),
    "investigation_summary": investigation_summary,
    "independent_direct_cross_check": direct_cross_check,
    "independent_direct_cross_check_all_within_tolerance": direct_cross_check_all_within_tolerance,
    "generated_at_utc": generated_at,
}
output_path = ARTIFACTS_DIR / "gate4_disparate_impact_investigation.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_record, f, indent=2)
print(f"\n[SAVED] {output_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 9: Append the additive config-yaml block (write_gate_block is generic and safe - does
# not touch the front matter, the `status` field, or any other existing gate block)
# ============================================================
from utils.bp1_config_sync import write_gate_block  # noqa: E402

investigation_marker = (
    "# --- Disparate-Impact Mitigation Investigation (governance addendum, appended, idempotent "
    "overwrite) ---"
)
recall_ratio_rounded = round(diagnosis["recall_ratio_min_over_max"], 6)
fpr_ratio_rounded = round(diagnosis["false_positive_rate_ratio_min_over_max"], 6)
recall_balanced_str = str(diagnosis["recall_balanced_four_fifths_style"]).lower()
fpr_balanced_str = str(diagnosis["false_positive_rate_balanced_four_fifths_style"]).lower()
verdict_category = diagnosis["verdict"].split(":")[0]
cross_check_passed_str = str(direct_cross_check_all_within_tolerance).lower()

investigation_block_lines = [
    "disparate_impact_mitigation_investigation:",
    f'  champion_model: "{CHAMPION_NAME}"',
    f"  source_adverse_impact_ratio_tags: {JSON_ADVERSE_IMPACT_RATIO}",
    f"  recall_ratio_min_over_max: {recall_ratio_rounded}",
    f"  false_positive_rate_ratio_min_over_max: {fpr_ratio_rounded}",
    f"  recall_balanced_four_fifths_style: {recall_balanced_str}",
    f"  false_positive_rate_balanced_four_fifths_style: {fpr_balanced_str}",
    f'  driver_verdict_category: "{verdict_category}"',
    f"  independent_direct_cross_check_all_within_tolerance: {cross_check_passed_str}",
    "  disparate_treatment_disclosure_included: true",
    "  auto_applied_to_production_model: false",
    # .as_posix() - the real Windows run of this notebook proved a real bug here: a bare
    # relative_to() on Windows yields backslash separators, and embedding those directly into a
    # double-quoted YAML string produces invalid escape sequences (e.g. "\g" is not a valid YAML
    # escape) that break yaml.safe_load on every later read of this config file. Forcing forward
    # slashes here is required, not cosmetic.
    f'  output_artifact: "{output_path.relative_to(PROJECT_ROOT).as_posix()}"',
    f'  generated_at_utc: "{generated_at}"',
]
write_gate_block(bp3_config_path, investigation_marker, investigation_block_lines)
print(
    f"[SAVED] {bp3_config_path.relative_to(PROJECT_ROOT)} (disparate_impact_mitigation_investigation block)"
)

# ============================================================
# SECTION 10: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "champion_matches_gate4_gate5_recorded": CHAMPION_NAME
    == gate4_block["champion_model"]
    == gate5_block["champion_model"],
    "config_and_json_adverse_impact_ratio_agree": abs(JSON_ADVERSE_IMPACT_RATIO - CONFIG_ADVERSE_IMPACT_RATIO)
    < 1e-9,
    "decision_records_row_count_matches_gate5_recorded": len(decision_records_df)
    == N_DECISION_RECORDS_RECORDED,
    "tags_group_values_match_gate5_breakdown": REAL_TAGS_GROUPS == BREAKDOWN_TAGS_GROUPS,
    "investigation_summary_has_all_required_keys": {
        "bp_id",
        "investigation",
        "source_adverse_impact_ratio_tags",
        "source_flagged",
        "group_confusion_detail",
        "diagnosis",
        "within_group_calibration",
        "equalized_fpr_threshold_simulation",
        "disparate_treatment_disclosure",
        "recommendation",
    }.issubset(investigation_summary.keys()),
    "disparate_treatment_disclosure_present_and_nonempty": bool(
        investigation_summary["disparate_treatment_disclosure"]
    )
    and investigation_summary["disparate_treatment_disclosure"] == DISPARATE_TREATMENT_DISCLOSURE,
    "diagnosis_verdict_is_nonempty_string": isinstance(diagnosis["verdict"], str)
    and len(diagnosis["verdict"]) > 0,
    "independent_direct_cross_check_passed_for_every_group": direct_cross_check_all_within_tolerance,
    "independent_direct_cross_check_covers_all_real_groups": set(direct_cross_check.keys())
    == REAL_TAGS_GROUPS,
    "output_artifact_written": output_path.exists(),
    "output_artifact_nonempty": output_path.stat().st_size > 0,
    "bp3_config_yaml_updated": bp3_config_path.exists(),
    "no_champion_model_retrained_or_changed": CHAMPION_NAME == "xgboost",
    "no_default_decision_threshold_changed": True,  # this notebook never writes a threshold override
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    f"\n[ALL CHECKS PASSED] BP3 disparate-impact mitigation investigation complete. Real "
    f"adverse_impact_ratio_tags={JSON_ADVERSE_IMPACT_RATIO} finding investigated using the full real "
    f"{len(decision_records_df):,}-row Gate 5 decision-records file. Diagnosis: "
    f"{diagnosis['verdict'].split(':')[0]}. Independent direct cross-check against the real "
    f"underlying rows PASSED for all {len(direct_cross_check)} real tags_group values. This "
    f"investigation does not retrain BP3's champion, does not change its Gate 3-5 real-run-"
    f"confirmed outputs, and does not change its default 0.5 decision threshold - it is additive "
    f"information for the human governance review this project's own Gate 4/5 flag already calls "
    f"for. Result written to {output_path.name} and appended to "
    f"{bp3_config_path.name}. NEXT STEP: this is information for the user's governance decision "
    f"on BP3 - see the disparate_treatment_disclosure and equalized_fpr_threshold_simulation "
    f"sections before deciding whether/how to act on it."
)
